# 01: Dataset Exploration and Histopathological Distribution Analysis

**Project:** Self-Supervised Representation Learning for Histopathological Image Classification Under Limited Label Availability  
**Dataset:** PathMNIST (MedMNIST v2)  
**Source Cohort:** NCT-CRC-HE-100K (Colorectal Cancer Histology) & CRC-VAL-HE-7K (External Test)

---

## 1. Environment Setup and Imports

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.datasets import PathMNISTDataset
from src.utils import load_config

config = load_config(os.path.join(project_root, 'configs', 'config.yaml'))
class_names = [config['dataset']['classes'][i] for i in range(config['dataset']['num_classes'])]
print("Classes:", class_names)

## 2. Dataset Split Sizes and Class Distributions

In [ ]:
data_path = os.path.join(project_root, 'data', 'pathmnist.npz')
data = np.load(data_path)

splits = ['train', 'val', 'test']
dist_records = []

for split in splits:
    labels = data[f'{split}_labels'].squeeze()
    unique, counts = np.unique(labels, return_counts=True)
    for u, c in zip(unique, counts):
        dist_records.append({
            'Split': split.capitalize(),
            'Class ID': int(u),
            'Class Name': class_names[int(u)],
            'Count': int(c)
        })

df_dist = pd.DataFrame(dist_records)

# Plot Class Distributions Across Splits
plt.figure(figsize=(12, 5))
sns.barplot(data=df_dist, x='Class Name', y='Count', hue='Split', palette='Set2')
plt.title('PathMNIST Class Distribution: Training, Validation, and External Test Splits', fontsize=13, fontweight='bold')
plt.xlabel('Tissue Type', fontsize=11)
plt.ylabel('Number of Patches', fontsize=11)
plt.xticks(rotation=40, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 3. Sample Patch Visualization Across 9 Tissue Classes

In [ ]:
train_images = data['train_images']
train_labels = data['train_labels'].squeeze()

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for c_id in range(9):
    ax = axes[c_id // 3, c_id % 3]
    sample_idx = np.where(train_labels == c_id)[0][0]
    img = train_images[sample_idx]
    ax.imshow(img)
    ax.set_title(f'Class {c_id}: {class_names[c_id]}', fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle('Representative Histological Tissue Patches (H&E Staining)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Empirical Channel Statistics (Mean and Standard Deviation)
Calculating the exact dataset normalization constants for RGB channels across the training cohort.

In [ ]:
# Convert uint8 [0, 255] to float32 [0.0, 1.0]
images_norm = train_images.astype(np.float32) / 255.0

channel_means = images_norm.mean(axis=(0, 1, 2))
channel_stds = images_norm.std(axis=(0, 1, 2))

print(f"Empirical Channel Means (R, G, B): {[round(x, 4) for x in channel_means]}")
print(f"Empirical Channel Stds  (R, G, B): {[round(x, 4) for x in channel_stds]}")